In [40]:
import csv
import json
from os import path

import librosa
import numpy as np
import pandas as pd
import torch
import torchaudio
from IPython.display import Audio
from sklearn.preprocessing import OrdinalEncoder
from torch.nn import functional as F
from torchaudio.transforms import MelSpectrogram
from torchcodec.decoders import AudioDecoder

from tspeech.model.tts import TTSModel
from tspeech.model.tacotron2.hifi_gan import Generator

DATASET_DIR = "/home/mmcneil/datasets/LibriTTS"

In [4]:
df = pd.read_csv(path.join(DATASET_DIR, "libritts-val.csv"), delimiter="|", quoting=csv.QUOTE_NONE)

In [9]:
df.speaker_idx.value_counts()

speaker_idx
425    46
825    44
138    39
375    37
158    36
       ..
876     1
822     1
276     1
745     1
740     1
Name: count, Length: 906, dtype: int64

In [53]:
model = TTSModel.load_from_checkpoint(
    "/home/mmcneil/src/Trustworthy_v2/lightning_logs/tacotron2-revised-code/version_22/checkpoints/checkpoint-epoch=34.ckpt"
)

# HiFi-GAN
class AttrDict(dict):
    def __init__(self, *args, **kwargs):
        super(AttrDict, self).__init__(*args, **kwargs)
        self.__dict__ = self


with open("UNIVERSAL_V1/config.json", "r") as infile:
    generator_config = AttrDict(json.load(infile))

generator_state_dict = torch.load(
    "UNIVERSAL_V1/g_02500000",
    map_location=f"cuda:0",
)

print(generator_config)
generator = Generator(generator_config)
generator.load_state_dict(generator_state_dict["generator"])

generator.remove_weight_norm()
generator.eval()
generator = generator.cuda()

Speaker tokens enabled with 928 speakers
{'resblock': '1', 'num_gpus': 0, 'batch_size': 16, 'learning_rate': 0.0002, 'adam_b1': 0.8, 'adam_b2': 0.99, 'lr_decay': 0.999, 'seed': 1234, 'upsample_rates': [8, 8, 2, 2], 'upsample_kernel_sizes': [16, 16, 4, 4], 'upsample_initial_channel': 512, 'resblock_kernel_sizes': [3, 7, 11], 'resblock_dilation_sizes': [[1, 3, 5], [1, 3, 5], [1, 3, 5]], 'segment_size': 8192, 'num_mels': 80, 'num_freq': 1025, 'n_fft': 1024, 'hop_size': 256, 'win_size': 1024, 'sampling_rate': 22050, 'fmin': 0, 'fmax': 8000, 'fmax_for_loss': None, 'num_workers': 4, 'dist_config': {'dist_backend': 'nccl', 'dist_url': 'tcp://localhost:54321', 'world_size': 1}}


/home/mmcneil/src/Trustworthy_v2/venv/lib/python3.14/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Removing weight norm...


In [22]:
chars_set = set()
for t in df.text:
    chars_set = chars_set.union(set(t))
chars = list(chars_set)
chars.sort()

In [23]:
encoder = OrdinalEncoder()
encoder.fit([[x] for x in chars + ["^"]])

melspectrogram = MelSpectrogram(
    sample_rate=16000,
    n_fft=1024,
    win_length=1024,
    hop_length=256,
    f_min=0.0,
    f_max=8000.0,
    n_mels=80,
    power=1.0,
    mel_scale="slaney",
    norm="slaney",
    center=True,
)

In [24]:
chars_idx = torch.tensor(
    encoder.transform([[x] for x in "hello, world" + "^"]) + 1, dtype=torch.int64
).swapaxes(0, 1)

# Load the random WAV for style
# wav, sr = torchaudio.load(path.join(DATASET_DIR, df.sample(1).iloc[0].wav))

with open(path.join(DATASET_DIR, df.sample(1).iloc[0].wav), "rb") as infile:
    decoder = AudioDecoder(infile.read(), sample_rate=16000)
    wav = decoder.get_all_samples().data

wav = wav.squeeze(0)

# Trimming
wav_np, _ = librosa.effects.trim(
    wav.numpy(),
    top_db=40,
    frame_length=1024,
)
wav = torch.tensor(wav_np)
wav = F.pad(wav, (0, 512))

# Generate the mel spectrogram
mel_spectrogram_style = melspectrogram(wav).swapaxes(0, 1)
mel_spectrogram_style = torch.log(
    torch.clamp(mel_spectrogram_style, min=1e-5)
).unsqueeze(0)
mel_spectrogram_style_len = torch.IntTensor([mel_spectrogram_style.shape[1]])

In [25]:
Audio(data=wav, rate=16000)

In [54]:
with torch.no_grad():
    mels, mels_post, gates, alignments = model(
        chars_idx.cuda(),
        torch.tensor([chars_idx.shape[1]]).cuda(),
        teacher_forcing=False,
        speaker_id=torch.tensor([825]).cuda(),
        mel_spectrogram_style=mel_spectrogram_style.cuda(),
        mel_spectrogram_style_len=mel_spectrogram_style_len.cuda(),
        max_len_override=400,
        teacher_forcing_dropout=0,
    )

end_idx = torch.argmax((gates[0].squeeze(-1) <= 0).type(torch.int)).item()
end_idx

101

In [55]:
out_wav = librosa.feature.inverse.mel_to_audio(
    np.exp(mels_post.cpu().numpy()[0, :end_idx].swapaxes(0, 1)),
    sr=16000,
    win_length=1024,
    hop_length=256,
    n_fft=1024,
    power=1.0,
)

In [56]:
with torch.no_grad():
    display(
        Audio(
            data=generator(mels_post[:, :end_idx].cuda().swapaxes(1, 2)).squeeze(0).cpu(),
            rate=20050,
        )
    )

In [57]:
Audio(data=out_wav, rate=16000)